## Calculating SDI and Delta SDI

This script was developed to calculate the Suppression Difficulty Index (SDI) and Delta SDI for various simulated fuel treatments. The SDI provides a rating of suppression difficulty, indicating how challenging it is to control a fire in a specific area by evaluating factors such as landscape characteristics, fuel types, and accessibility.

Delta SDI quantifies the change in SDI resulting from simulated fuel treatments. It compares the SDI derived from a baseline fuel simulation—where surface fuels have not been modified—with the SDI obtained from a scenario in which the fuel surface is theoretically treated to reduce fire risk. 

In the baseline simulation, the SDI reflects the current state of the landscape as represented by the LANDFIRE data used in the analysis. In contrast, the treated scenario represents a landscape that has been theoretically altered through the implementation of strategic fuel reduction methods, including rx burns, mechanical thinning, and various other forest management practices.

#### Import libraries

In [ ]:
import arcpy, os, sys, traceback, shutil, numpy as np
from arcpy.sa import *

#### Calling in datasets
Import all datasets required for completing the SDI and Delta SDI modeling. This includes topographic data such as elevation and aspect, as well as transportation data like streets and trails. Additionally, a lookup table referenced in the SDI analysis is also included.

In [ ]:
#read in clipped LANDFIRE data from per-processing
rasters_file_path = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Test\Adjust_Modify_Fuelscapes\Analysis_Data\Clipped_Landfire_Data' #change this to match your directory structure

INelve = arcpy.Raster(os.path.join(rasters_file_path, 'ELEV.tif'))
INasp = arcpy.Raster(os.path.join(rasters_file_path, 'ASP.tif'))

#read in transportation datasets
#this has to be HERE data as the SDI analysis relies on specific attribute names
streets = r'C:\Users\AlexHeeren\CFRI\Data\Reference_National_Dataset\NationalDatasets\Transportation\HERE_FSroads\Western_Rds.gdb\HereRds' #change this to match your directory structure
trails = r'C:\Users\AlexHeeren\CFRI\Data\Reference_National_Dataset\NationalDatasets\Transportation\USFS_Trails_2024.gdb\USFS_Trails' #change this to match your directory structure

#specify the file paths for FBFM40, Flame Length, and Heat Unit Area
#this path points to the folder where all the intermediate simulated fuel treatments for each scenario were created, with the FBFM40 generated for the corresponding simulated fuel treatment
fbfmpath = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Test\Adjust_Modify_Fuelscapes\Outputs\Intermediate_Simulate_Fuel_Treatment' #change this to match your directory structure

#this path points to the folder where all fire modeling outputs for flame length and heat unit area were generated for each of the simulated treatment
fbpath = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Test\FlamMap_Modeling\Outputs' #change this to match your directory structure

#read in lookup table
rtc_lookup = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Scripts_Supporting_Files\SDI\08_RTC_lookup_SDIwt_westernUS_2021_update.txt' #change this to match your directory structure

#### Set environments

In [ ]:
from arcpy import env #setting the workspace and environment
workspace = r'C:\Users\AlexHeeren\CFRI\All_Projects\Fire_Modeling\Test' #change this to match your directory structure

#set up scratch workspace
#the necessary folders and .gdb file will be created in the folder structure setup section below.
arcpy.env.scratchWorkspace = workspace + r'\SDI\SDI_Calc\Scratch.gdb' 

#input spatial reference
#NAD83 UTM Zone12 = 26912|NAD83 UTM Zone13 = 26913|North American Albers Equal Area Conic = 102008
env.outputCoordinateSystem = arcpy.SpatialReference(26913) #set spatial reference 

env.overwriteOutput = True #sets outputs to overwrite

arcpy.env.addOutputsToMap = False #stops outputs from being displayed in TOC

#set up ArcPy environment settings for raster processing to ensure consistent outputs
arcpy.env.snapRaster = INasp #snap raster outputs to ensure that all datasets are aligned with each other
arcpy.env.extent = INasp #set entexnt

#### Set up folder structure
Create folders to organize the data prepared by this script and maintain a standardized folder structure, which will facilitate integration with other scripts for fuelscape modification and fire modeling. This process also includes establishing a scratch workspace designed for the intermediate data generated during the SDI analysis.

In [ ]:
#create the SDI folder as the root directory for all data related to SDI analysis
root_folder = os.path.join(workspace, 'SDI')
if not os.path.exists(root_folder):
    os.makedirs(root_folder)

#create Delta_SDI folder to store all final delta SDI outputs 
delta_sdi_folder = os.path.join(root_folder, 'Delta_SDI')
if not os.path.exists(delta_sdi_folder):
    os.makedirs(delta_sdi_folder)

#create SDI_Calc folder to store all SDI calculations data
sdi_folder = os.path.join(root_folder, 'SDI_Calc')
if not os.path.exists(sdi_folder):
    os.makedirs(sdi_folder)
    
#create Outputs folder to store all SDI calculations data for each simulated fuel treatment
outputs_folder = os.path.join(sdi_folder, 'Outputs') 
if not os.path.exists(outputs_folder):
    os.makedirs(outputs_folder)
    
#create a .gdb file for the scratch workspace to store intermediate data generated during the SDI analysis
arcpy.CreateFileGDB_management(sdi_folder, 'Scratch')

#### Create variables
Prepares the necessary variables and file paths for conducting SDI and Delta SDI analyses. It defines user inputs such as the weather percentile, project area, and simulated fuel treatments types. Additionally, generates a standardized output file name based on the weather percentile and area name.

In [ ]:
#user input
#choose the weather percentile of interest from the Delta SDI analysis
weather_pct = '90' #replace '90' with the desired weather percentile

#define the project or area name for easier identification of the output files
area_name = 'RENAME' #replace with the project or area name

#create the base string for the output .tif name, combining the weather percentile and area name
oname_base = 'sdi_Pct' + str(weather_pct) + '_' + str(area_name)

#choose the simulated fuel treatment(s) on which to run the SDI and Delta SDI analyses
#list the treatments you want to include available treatments for reference:
#'baseline', 'hand_thin', 'mech_thin', 'rx_fire', 'hand_complete', 'mech_complete', 'patch_cut', 'patch_cut_sm', 'mast'
#copy paste treatment type of intrest to run anaylsis on
trts = ['mech_thin']  #replace with one or more treatments you want to run

#### SDI Analysis
This section of the script performs the Suppression Difficulty Index (SDI) analysis.

The code has been modified by Alex Heeren to meet the specific requirements of the Delta SDI analysis, with the primary change being the addition of a for loop to iterate through each simulated treatment. The original version was developed by Ben Gannon, and any further questions regarding the initial implementation can be directed to him.

This script models the Suppression Difficulty Index (SDI) as outlined by Rodríguez y Silva et al. (2020), with an additional universal slope modification to enhance accuracy. It is based on previous work by Matt Panunto, Quresh Latif, and Pyrologix, reflecting continuous improvements in SDI modeling for fire management applications.

In [ ]:
#####-----> Main body of analysis
print('####----> Suppression Difficulty Index')
print

#First Calculate static indices

#-> ACCESSIBILITY SUB INDEX
# Pyrologix edits:
# 1. Run Euclidean Distance on the streets layer
# 2. Classify areas within 100 m of a road as a 10, decreasing to 1 where the nearest road is >900 m away
print('#-> Calculating accessibility sub index')
streets_lyr = arcpy.MakeFeatureLayer_management(streets,'streets_lyr')
arcpy.SelectLayerByAttribute_management(streets_lyr,'NEW_SELECTION',where_clause='"SpeedCat" < 8')
print('Selected roads with speedclass < 8')
roaddist = EucDistance(streets_lyr,'',30)
print('Calculated distance to roads')
accessibility = Con((roaddist <= 100),10,
                Con((roaddist > 100) & (roaddist <= 200), 9,
                Con((roaddist > 200) & (roaddist <= 300), 8,
                Con((roaddist > 300) & (roaddist <= 400), 7,
                Con((roaddist > 400) & (roaddist <= 500), 6,
                Con((roaddist > 500) & (roaddist <= 600), 5,
                Con((roaddist > 600) & (roaddist <= 700), 4,
                Con((roaddist > 700) & (roaddist <= 800), 3,
                Con((roaddist > 800) & (roaddist <= 900), 2,
                1)))))))))

#accessibility.save(arcpy.env.scratchWorkspace + r'\accessibility')
print('Reclassified distance to roads raster to accessibility index value')
print

#-> MOBILITY SUB-INDEX
# Pyrologix edit: setting mobility = 1

#-> PENETRABILITY SUB-INDEX
# Pyrologix edits:
# 1. use focal mean (56.41896m) rather than prop fuel
print('#-> Calculating penetrability sub index')

# Convert Aspect Raster (degrees) into assigned values
# Changed from the paper after 2018 Cordoba visit
aspect_class = Con((INasp >= 337.5), 10, # North Facing
                   Con((INasp >= -1)    & (INasp < 22.5), 10, # North Facing
                   Con((INasp >= 22.5)  & (INasp < 67.5), 8,  # Northeast Facing
                   Con((INasp >= 292.5) & (INasp < 337.5), 7, # Northwest Facing
                   Con((INasp >= 67.5)  & (INasp < 112.5), 6, # East Facing
                   Con((INasp >= 247.5) & (INasp < 292.5), 5, # West Facing
                   Con((INasp >= 112.5) & (INasp < 157.5), 4, # Southeast Facing
                   Con((INasp >= 202.5) & (INasp < 247.5), 3, # Southwest Facing
                   Con((INasp >= 157.5) & (INasp < 202.5), 1))))))))) # South Facing 
print('Converted aspect raster into assigned values')

# Calculate Percent Slope
slope = Slope(INelve,'PERCENT_RISE')
print('Calculated percent slope')

# Convert Percent Slope Raster into assigned values
slope_class = Con((slope >= 0) & (slope < 6), 10,
                  Con((slope >= 6) & (slope < 11), 9,
                  Con((slope >= 11) & (slope < 16), 8,
                  Con((slope >= 16) & (slope < 21), 7,
                  Con((slope >= 21) & (slope < 26), 6,
                  Con((slope >= 26) & (slope < 31), 5,
                  Con((slope >= 31) & (slope < 36), 4,
                  Con((slope >= 36) & (slope < 41), 3,
                  Con((slope >= 41) & (slope < 46), 2,
                  Con((slope >= 46), 1))))))))))
print('Converted percent slope into assigned values')


# Merge streets speed cat 8 with trails
streets_lyr = arcpy.MakeFeatureLayer_management(streets,'streets_lyr')
arcpy.SelectLayerByAttribute_management(streets_lyr,'NEW_SELECTION',where_clause='"SpeedCat" > 7')
trails_streets = arcpy.env.scratchWorkspace + r'\trails_streets'
arcpy.Merge_management([trails,streets_lyr],trails_streets)
print('Merged street speed cat 8 with trails')

# Calculate length of trails within 1 ha (56.41896m radius) moving window.
trail_len = LineStatistics(trails_streets,'NONE','30','56.41896','LENGTH')
print('Calculated length of trails within 1 ha')

# Convert pre-suppression trails raster into assigned values
trail_len_class = Con((trail_len >= 0) & (trail_len < 10), 1,
                      Con((trail_len >= 10) & (trail_len < 20), 2,
                      Con((trail_len >= 20) & (trail_len < 30), 3,
                      Con((trail_len >= 30) & (trail_len < 40), 4,
                      Con((trail_len >= 40) & (trail_len < 50), 5,
                      Con((trail_len >= 50) & (trail_len < 60), 6,
                      Con((trail_len >= 60) & (trail_len < 70), 7,
                      Con((trail_len >= 70) & (trail_len < 80), 8,
                      Con((trail_len >= 80) & (trail_len < 90), 9,
                      Con((trail_len >= 90), 10))))))))))
print('Reclassified road/trail lengths to assigned values')

### Loop through necceasry dynamic calculations

for y in range(len(trts)):

    print('\nCalculating SDI for ' + trts[y] + '...')

    # Set dynamic file paths
    FBFM40 = fbfmpath + '\\' + trts[y] + '\\fbfm.tif'
    FL = fbpath + '\\' + trts[y] + '_Pct' + str(weather_pct) + '_FLAMELENGTH.tif'
    HUA = fbpath + '\\' + trts[y] + '_Pct' + str(weather_pct) + '_HEATAREA.tif'
    FBFM40 = Raster(FBFM40)
    FL = Raster(FL)
    HUA = Raster(HUA)
    oname = trts[y] + '_' + str(oname_base) + '.tif'
    print('Set dynamic file paths')


    #-> ENERGY BEHAVIOR SUB-INDEX
    print('#-> Calculating Energy Behavior Sub Index')

    # Reclassify Flammap flamelength raster values based on Table 2 of
    # Rodriguez y Silva et al
    # Ensure that units are first converted to meters before reclassifying,
    # if necessary.
    FL_recl = Con((FL <= 0.5), 1,
                  Con((FL > 0.5) & (FL <= 1), 2,
                  Con((FL > 1)   & (FL <= 1.5), 3,
                  Con((FL > 1.5) & (FL <= 2), 4,
                  Con((FL > 2)   & (FL <= 2.5), 5,
                  Con((FL > 2.5) & (FL <= 3), 6,
                  Con((FL > 3)   & (FL <= 3.5), 7,
                  Con((FL > 3.5) & (FL <= 4), 8,
                  Con((FL > 4)   & (FL <= 4.5), 9,
                  Con((FL > 4.5), 10))))))))))
    print('Reclassified flame lengths to assigned index values')

    # Reclassify Flammap heat per unit area raster values based on
    # Table 2 of Rodriguez y Silva et al
    # Convert heat per unit area raster from kJ/m2 to kcal/m2
    HUA_kcal = HUA * 0.2388459
    HUA_recl = Con((HUA_kcal <= 380),1,
                   Con((HUA_kcal > 380)  & (HUA_kcal <= 1265), 2,
                   Con((HUA_kcal > 1265) & (HUA_kcal <= 1415), 3,
                   Con((HUA_kcal > 1415) & (HUA_kcal <= 1610), 4,
                   Con((HUA_kcal > 1610) & (HUA_kcal <= 1905), 5,
                   Con((HUA_kcal > 1905) & (HUA_kcal <= 2190), 6,
                   Con((HUA_kcal > 2190) & (HUA_kcal <= 4500), 7,
                   Con((HUA_kcal > 4500) & (HUA_kcal <= 6630), 8,
                   Con((HUA_kcal > 6630) & (HUA_kcal <= 8000), 9,
                   Con((HUA_kcal > 8000), 10))))))))))
    print('Reclassified heat per unit area to assigned index values')

    # Calculate Energy Behavior raster
    # The focal statistics smoothing operation replaces the previous propFuel multplication factor
    eb_rast = (2 * Float(FL_recl) * Float(HUA_recl) / (Float(FL_recl) + Float(HUA_recl)))
    print('Created energy behavior raster')
    # Smooth, but set non-burnable pixels to value of 1
    neighborhood = NbrCircle(56.41896,'MAP')
    eb_smooth = Con(FBFM40 > 100,FocalStatistics(eb_rast,neighborhood,'MEAN','DATA'),1)
    #eb_smooth.save(arcpy.env.scratchWorkspace + r'\energy_behavior')
    print('Smoothed energy behavior raster')
    print


    # Reclassify fuels into assigned RTC values using reclass table
    # This will convert the fuel types into a fuel control difficulty weight based on estimated
    # fireline production rates (meters per mile) for a 20-person crew. Note that this is an
    # approximation of the original table in Rodriguez y Silva et al 2014 for Scott and Burgan
    # (2005) fuel models.
    fuel_cntrl = ReclassByASCIIFile(FBFM40,rtc_lookup,'NODATA')
    print('Reclassified fuels into assigned fireline production values')

    # Create raster for Penetrability Sub Index
    # The focal statistics smoothing operation replaces the previous propFuel multplication factor
    penetrability = (Float(slope_class) + Float(fuel_cntrl) + Float(aspect_class) + (2*Float(trail_len_class)))/5
    print('Calculated penetrability index')
    neighborhood = NbrCircle(56.41896,'MAP')
    pen_smooth = FocalStatistics(penetrability,neighborhood,'MEAN','DATA')
    #pen_smooth.save(arcpy.env.scratchWorkspace + r'\penetrability')
    print('Smoothed penetrability index')
    print


    #-> FIRELINE OPENING/CREATION SUB-INDEX
    print('#-> Calculating fireline opening/creation sub index')

    # Create Slope Adjustment Raster for hand work
    slope_adjust_hand = Con((slope >= 0) & (slope < 16), 1,
                            Con((slope >= 16) & (slope < 31), 0.8,
                            Con((slope >= 31) & (slope < 46), 0.6,
                            Con((slope >= 46), 0.5))))
    print('Calculated slope adjustment raster for hand work')

    # Create Slope Adjustment Raster for machine work
    slope_adjust_mach = Con((slope >= 0) & (slope < 16), 1,
                            Con((slope >= 16) & (slope < 21), 0.8,
                            Con((slope >= 21) & (slope < 26), 0.7,
                            Con((slope >= 26) & (slope < 31), 0.6,
                            Con((slope >= 31) & (slope < 36), 0.5,
                            Con((slope >= 36), 0))))))
    print('Calculated slope adjustment raster for machine work')

    # Calculate Fireline Opening Sub Index
    firelineopening = (Float(fuel_cntrl)*slope_adjust_hand) + (Float(fuel_cntrl)*slope_adjust_mach)
    #firelineopening.save(arcpy.env.scratchWorkspace + r'\firelineopening')
    print('Calculated fireline opening index')
    print

    #-> Kit O'Connor (KO) UNIVERSAL SLOPE MOBILITY HAZARD ADJUSTMENTS
    print('#-> Calculating KO universal slope mobility hazard adjustment')

    # Create slope hazard raster    
    slope_haz = Con((slope >= 0) & (slope < 10), 0.01,
                    Con((slope >= 10) & (slope < 20), 0.1,
                    Con((slope >= 20) & (slope < 30), 0.2,
                    Con((slope >= 30) & (slope < 40), 0.4,
                    Con((slope >= 40) & (slope < 50), 0.6,   
                    Con((slope >= 50) , 0.8,))))))
    print('Calculated slope mobility hazard')
    print

    #-> SDI CALCULATION
    print('#-> Calculing SDI from sub-indices')

    # Create SDI raster without slope modification
    # mobility = 1, 2020 publication version
    SDI = Float(eb_smooth) / (Float(accessibility) + Float(1) + Float(pen_smooth) + Float(firelineopening))
    SDI = Con(IsNull(SDI) & (FBFM40 < 100),Float(0),SDI) # Setting non-burnable pixels to zero
    print('Calculated SDI without slope modification')

    # Optional: save denominator for inspection
    #sdi_denom = (Float(accessibility) + Float(1) + Float(pen_smooth) + Float(firelineopening))
    #sdi_denom.save(arcpy.env.scratchWorkspace + r'\denominator')

    # Create SDI raster with universal slope modification
    SDI_SLP = SDI + slope_haz
    print('Calculated SDI with slope modification')

    # Set water to zero SDI
    SDI_SLP_H2O = Con(FBFM40==98,Float(0),SDI_SLP)
    print('Set water to zero SDI')

    # Combine using base model for roads & trails, and slope modification for everything else
    trails_rds = arcpy.env.scratchWorkspace + r'\trails_st_all'
    arcpy.Merge_management([trails,streets],trails_rds)
    arcpy.AddField_management(trails_rds,'ZONE','SHORT')
    arcpy.CalculateField_management(trails_rds,'ZONE','1','PYTHON_9.3','')
    trails_rds_rast = arcpy.env.scratchWorkspace + r'\trails_st_all_rast'
    arcpy.FeatureToRaster_conversion(trails_rds,'ZONE',trails_rds_rast,SDI_SLP)
    fSDI = Con(IsNull(trails_rds_rast),SDI_SLP_H2O,SDI)
    print('Combined base and slope adjusted SDI for roaded/trailed or unroaded/untrailed areas')
    print

    # Save as integer
    fSDI100 = Int(fSDI*Float(100))
    fSDI100.save(outputs_folder + '\\' + trts[y] + '_' + str(oname_base) + '.tif')
    print('Saved as integer')
    print
    print('Finished Calculating SDI for ' + trts[y] + '\n')
    
print("\nAll Done Calculating SDI")

#### Calculate Delta SDI
Identifies the baseline raster file by searching for 'baseline.tif within the output folder. It then iterates through the other rasters, to compute the Delta SDI. The Delta SDI is calculated by subtracting the baseline SDI from each treatment raster, and the result is saved in the specified Delta SDI output folder.

In [ ]:
#identifies and finds the baseline raster file by searching for the first .tif file containing 'baseline' in its name

#initialize a variable to store the baseline raster file name
baseline_file = None
#iterate through all files in the specified output folder
for filename in os.listdir(outputs_folder):
    #check if the filename contains 'baseline.tif'
    if 'baseline' in filename and filename.endswith('.tif'):
        #add to the baseline_file variable
        baseline_file = filename
        break  #stop searching once we find the baseline file

#check if the baseline file was found
if baseline_file is None:
    raise FileNotFoundError("Baseline raster NOT found in the output folder")

#iterate through all raster files in the specified output folder
for raster in os.listdir(outputs_folder):
    #check if the file is a .tif file and is not the baseline file
    if raster.endswith(".tif") and raster != baseline_file:
        print(f"Processing {raster}...")
        #calculate the Delta SDI by subtracting the baseline SDI from the SDI of the current raster
        baseline_raster = arcpy.sa.Raster(os.path.join(outputs_folder, baseline_file))
        current_raster = arcpy.sa.Raster(os.path.join(outputs_folder, raster))
        result = baseline_raster - current_raster
        
        #construct the unique output file name 
        output_name = raster.replace('.tif', '')  #remove the .tif extension
        parts = output_name.split('_')  #split the name by underscores
        sdi_index = parts.index('sdi')  #find the index of 'sdi' in the list
        parts.insert(sdi_index, 'delta')  #insert 'delta' before 'sdi'
        output_name_with_delta = '_'.join(parts) + '.tif'  #join the parts back with underscores
        
        result.save(os.path.join(delta_sdi_folder, output_name_with_delta))
        print(f"Processed {raster}\n")

print("\nAll Done Calculating Delta SDI")